# 变化检测

## 介绍

变化检测比较不同时间拍摄的同一地理区域的图像，并识别发生有意义差异的位置。它广泛用于城市规划、环境保护、灾害响应和气候监测。

这一实践具有挑战性，因为不同日期的图像不仅因真实表面变化而不同，还因太阳角度、大气条件和植被物候的变化而不同。传统方法如图像差分和变化矢量分析通过像素级算法检测变化，而深度学习模型学习空间模式以区分有意义的变化与混淆的辐射变化。

本教程涵盖Landsat图像的传统变化检测方法，然后介绍使用`geoai`包中的ChangeStar模型进行基于深度学习的变化检测。您将在NAIP航空图像上应用ChangeStar，比较模型变体，调整检测阈值，并导出地理参考结果。

## 学习目标

在本教程结束时，您将能够：

- 解释双时相变化检测的原理并将其与多时相分析区分开来
- 应用传统的变化检测方法，包括图像差分和变化矢量分析
- 描述孪生网络如何通过比较图像对的学习特征表示来检测变化
- 使用`geoai`包在航空图像上运行ChangeStar变化检测
- 解释ChangeStar的三个输出：变化图、T1建筑分割和T2建筑分割
- 比较不同的ChangeStar模型变体并调整概率阈值以控制检测灵敏度
- 将变化检测结果导出为地理参考的GeoTIFF和矢量多边形，用于GIS集成

## 理解变化检测

变化检测生成突出显示表面变化位置的地图。输出可以是二值掩码（变化与未变化）、类别地图（变化类型）或连续表面（变化程度）。

### 变化类型

**突发变化**突然发生并产生显著的光谱差异，例如建筑物拆除、野火或洪水。

**渐进变化**在数月或数年内展开，例如城市扩张、海岸侵蚀或冰川退缩。检测它们通常需要长时间序列。

**季节性变化**是周期性变化，如落叶或作物收割。一个稳健的系统必须将这些与真实表面变化区分开来，通常通过比较不同年份同一季节的图像。

**永久性与临时性变化**对于解释很重要。新建建筑是永久性的；被洪水淹没的田地可能在几周内恢复正常。

### 挑战

- **几何错位**：如果两个图像没有精确配准，每个错位的像素都会表现为虚假变化。
- **大气和光照效应**：雾霾、云阴影和太阳角度的差异会引入与表面变化无关的辐射变化。
- **季节性和物候变化**：森林在夏季和冬季看起来非常不同，造成与感兴趣变化无关的光谱差异。
- **混合像素**：在较粗分辨率下，像素内的小变化可能太细微而无法检测。

## 传统变化检测方法

这些无监督方法作为基线、快速分析以及在没有标注数据时仍然有用。

### 图像差分

从一个日期的像素值中减去另一个日期的像素值，然后对结果进行阈值处理。具有大绝对差异的像素被分类为变化像素。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geoai

# Download Landsat imagery for two dates
url_2023 = "https://data.source.coop/opengeos/geoai/knoxville_landsat_2023.tif"
url_2024 = "https://data.source.coop/opengeos/geoai/knoxville_landsat_2024.tif"
path_2023 = geoai.download_file(url_2023)
path_2024 = geoai.download_file(url_2024)

# Read the NIR band (Band 5 in Landsat 8/9) from both dates
with rasterio.open(path_2023) as src:
    nir_2023 = src.read(5).astype(np.float32)

with rasterio.open(path_2024) as src:
    nir_2024 = src.read(5).astype(np.float32)

# Compute the difference
diff = nir_2024 - nir_2023

# Threshold to identify significant changes
threshold = 2 * np.std(diff)
change_mask = np.abs(diff) > threshold

print(f"Difference range: {diff.min():.2f} to {diff.max():.2f}")
print(f"Threshold: {threshold:.2f}")
print(f"Changed pixels: {change_mask.sum():,} ({100 * change_mask.mean():.1f}%)")

```text
差异范围：-0.23 到 0.48
阈值：0.02
变化像素：16,286（3.6%）
```

阈值选择至关重要——太低会导致噪声充斥地图；太高会错过真实变化。常用方法包括标准差倍数、Otsu方法或经验调整。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(diff, cmap="RdBu", vmin=-threshold * 2, vmax=threshold * 2)
axes[0].set_title("NIR Difference (2024 - 2023)")
axes[0].axis("off")
axes[1].hist(diff.ravel(), bins=100, color="steelblue", edgecolor="none")
axes[1].axvline(
    -threshold, color="red", linestyle="--", label=f"Threshold (±{threshold:.2f})"
)
axes[1].axvline(threshold, color="red", linestyle="--")
axes[1].set_title("Difference Histogram")
axes[1].legend()
axes[2].imshow(change_mask, cmap="Reds")
axes[2].set_title(f"Change Mask ({100 * change_mask.mean():.1f}% changed)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

### 变化矢量分析

变化矢量分析（CVA）将图像差分扩展到多个波段。它将每个像素视为光谱空间中的矢量，并计算日期之间变化的幅度和方向。

In [ ]:
# Read Red and NIR bands from both dates
with rasterio.open(path_2023) as src:
    red_2023 = src.read(4).astype(np.float32)
    nir_2023 = src.read(5).astype(np.float32)
with rasterio.open(path_2024) as src:
    red_2024 = src.read(4).astype(np.float32)
    nir_2024 = src.read(5).astype(np.float32)

delta_red = red_2024 - red_2023
delta_nir = nir_2024 - nir_2023
magnitude = np.sqrt(delta_red**2 + delta_nir**2)
direction = np.degrees(np.arctan2(delta_nir, delta_red))

mag_threshold = np.percentile(magnitude, 95)
significant_change = magnitude > mag_threshold
print(f"CVA magnitude range: {magnitude.min():.2f} to {magnitude.max():.2f}")
print(f"95th percentile threshold: {mag_threshold:.2f}")
print(f"Significant change pixels: {significant_change.sum():,}")

```text
CVA幅度范围：0.00 到 0.68
第95百分位阈值：0.03
显著变化像素：22,910
```

方向分量提供了关于变化性质的信息。例如，在红-NIR空间中，从植被到裸土的转变产生指向特征方向的变化矢量。

## 用于变化检测的深度学习

深度学习模型从标注示例中学习空间模式和上下文信息，使它们能够区分有意义的表面变化与无关的辐射变化。

### 孪生网络

孪生网络通过具有共享权重的并行编码器分支处理两个图像。两个分支在同一表示空间中提取特征，比较模块识别相应空间位置的差异。著名的架构包括FC-Siam-diff、FC-Siam-conc和BIT（使用Transformer的二值变化检测）。

### ChangeStar

ChangeStar联合执行变化检测和语义分割。它产生三个输出：变化掩码，以及两个时间步的建筑分割图。该模型使用Changen2预训练权重，这些权重通过变化生成策略学习，无需微调即可很好地推广到新区域。`geoai`包通过`ChangeStarDetection`类集成了ChangeStar。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 导入库

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import geoai
from geoai.change_detection import (
    ChangeStarDetection,
    changestar_detect,
    list_changestar_models,
)

## 列出可用模型

ChangeStar提供多个模型变体。命名约定反映了预训练阶段和骨干架构（`vitb`表示ViT-Base，`vitl`表示ViT-Large）。

In [ ]:
models = list_changestar_models()
for short_name, full_name in models.items():
    print(f"{short_name:30s} -> {full_name}")

```text
s0_s1c1_vitb                   -> s0_init_s1c1_changestar_vitb_1x256
s0_s1c5_vitb                   -> s0_init_s1c5_changestar_vitb_1x256
s0_s9c1_vitb                   -> s0_init_s9c1_changestar_vitb_1x256
s0_xview2_s1c5_vitb            -> s0_init_xView2_ft_s1c5_changestar_vitb_1x256
s1_s1c1_vitb                   -> s1_init_s1c1_changestar_vitb_1x256
s1_s1c1_vitl                   -> s1_init_s1c1_changestar_vitl_1x256
s9_s9c1_vitb                   -> s9_init_s9c1_changestar_vitb_1x256
```

## 设置

检查可用的计算设备并创建输出目录。

In [ ]:
device = geoai.get_device()
print(f"Using device: {device}")

out_folder = "changestar_results"
Path(out_folder).mkdir(exist_ok=True)
print(f"Output directory: {out_folder}")

## 下载示例数据

我们使用2019年和2022年拍摄的拉斯维加斯NAIP航空图像。拉斯维加斯在此期间经历了快速的郊区发展，使其成为建筑变化检测的良好测试案例。

In [ ]:
naip_2019_url = "https://data.source.coop/opengeos/geoai/las_vegas_naip_2019_a.tif"
naip_2022_url = "https://data.source.coop/opengeos/geoai/las_vegas_naip_2022_a.tif"

naip_2019_path = geoai.download_file(naip_2019_url)
naip_2022_path = geoai.download_file(naip_2022_url)

print(f"Downloaded 2019 NAIP: {naip_2019_path}")
print(f"Downloaded 2022 NAIP: {naip_2022_path}")

## 可视化输入图像

创建2019年和2022年NAIP图像的分割地图以检查研究区域。

In [ ]:
geoai.create_split_map(
    left_layer=naip_2019_path,
    right_layer=naip_2022_path,
    left_label="NAIP 2019",
    right_label="NAIP 2022",
)

## 初始化ChangeStar模型

使用`s1_s1c1_vitb`变体（带有阶段1 Changen2预训练的ViT-Base骨干）创建`ChangeStarDetection`实例。权重在首次使用时自动下载。

In [ ]:
detector = ChangeStarDetection(model_name="s1_s1c1_vitb")

## 运行变化检测

`predict`方法接受两个GeoTIFF路径，并返回包含二值变化图、变化概率和分割结果的字典。大图像自动进行切片处理并使用重叠平均。

In [ ]:
result = detector.predict(
    naip_2019_path,
    naip_2022_path,
    output_change=os.path.join(out_folder, "change_map.tif"),
    output_t1_semantic=os.path.join(out_folder, "t1_buildings.tif"),
    output_t2_semantic=os.path.join(out_folder, "t2_buildings.tif"),
    output_vector=os.path.join(out_folder, "changes.gpkg"),
)

检查结果字典。

In [ ]:
print("Result keys:", list(result.keys()))
for key, value in result.items():
    if hasattr(value, "shape"):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"  {key}: {value}")

```text
结果键：['change_map', 'change_prob', 't1_semantic', 't2_semantic', 't1_semantic_prob', 't2_semantic_prob', 'num_semantic_classes']
  change_map: shape=(1419, 2721), dtype=uint8
  change_prob: shape=(1419, 2721), dtype=float32
  t1_semantic: shape=(1419, 2721), dtype=uint8
  t2_semantic: shape=(1419, 2721), dtype=uint8
  t1_semantic_prob: shape=(1419, 2721), dtype=float32
  t2_semantic_prob: shape=(1419, 2721), dtype=float32
  num_semantic_classes: 1
```

## 可视化结果

`visualize`方法显示一个五面板图，显示前后图像、变化图和两个时间步的建筑分割。

In [ ]:
fig = detector.visualize(
    naip_2019_path,
    naip_2022_path,
    result=result,
    figsize=(25, 10),
    title1="NAIP 2019",
    title2="NAIP 2022",
)
plt.show()

### 叠加可视化

`visualize_overlay`方法显示叠加在原始图像上的建筑分割（蓝色）和变化（红色）。

In [ ]:
fig = detector.visualize_overlay(
    naip_2019_path,
    naip_2022_path,
    result=result,
    figsize=(20, 6),
    title1="NAIP 2019",
    title2="NAIP 2022",
)
plt.show()

## 使用便捷函数

对于快速一次性分析，`changestar_detect()`初始化模型、运行预测并在单次调用中返回结果。

In [ ]:
result2 = changestar_detect(
    naip_2019_path,
    naip_2022_path,
    model_name="s1_s1c1_vitb",
    output_change=os.path.join(out_folder, "change_map_v2.tif"),
)

print(f"Change pixels: {result2['change_map'].sum():,}")
print(
    f"Changed area: {result2['change_map'].sum() / result2['change_map'].size * 100:.2f}%"
)

## 比较模型变体

不同的ChangeStar变体使用不同的预训练策略。这里我们比较`s1`和`s9`变体。

In [ ]:
result_s1 = result

detector_s9 = ChangeStarDetection(model_name="s9_s9c1_vitb")
result_s9 = detector_s9.predict(naip_2019_path, naip_2022_path)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(result_s1["change_map"], cmap="gray")
axes[0].set_title(f"S1 Model\n(Changed pixels: {result_s1['change_map'].sum():,})")
axes[0].axis("off")

axes[1].imshow(result_s9["change_map"], cmap="gray")
axes[1].set_title(f"S9 Model\n(Changed pixels: {result_s9['change_map'].sum():,})")
axes[1].axis("off")

combined = np.zeros((*result_s1["change_map"].shape, 3), dtype=np.uint8)
combined[result_s1["change_map"] == 1, 0] = 255  # S1 in red
combined[result_s9["change_map"] == 1, 2] = 255  # S9 in blue
both = (result_s1["change_map"] == 1) & (result_s9["change_map"] == 1)
combined[both] = [255, 0, 255]

axes[2].imshow(combined)
axes[2].set_title("Comparison\n(Red=S1 only, Blue=S9 only, Magenta=Both)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 调整阈值

默认概率阈值为0.5。降低它会提高灵敏度（更多检测，更多误报）；提高它会提高特异性（更少检测，更高置信度）。

In [ ]:
thresholds = [0.3, 0.5, 0.7]
fig, axes = plt.subplots(1, len(thresholds), figsize=(18, 6))

for ax, thresh in zip(axes, thresholds):
    result_t = detector.predict(naip_2019_path, naip_2022_path, threshold=thresh)
    ax.imshow(result_t["change_map"], cmap="gray")
    ax.set_title(
        f"Threshold = {thresh}\n" f"(Changed pixels: {result_t['change_map'].sum():,})"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 查看保存的输出

保存的GeoTIFF和GeoPackage可以直接加载到任何GIS软件中。

In [ ]:
for f in sorted(os.listdir(out_folder)):
    fpath = os.path.join(out_folder, f)
    size_mb = os.path.getsize(fpath) / 1024 / 1024
    print(f"{f:40s} {size_mb:.2f} MB")

```text
change_map.tif                           0.11 MB
change_map_v2.tif                        0.11 MB
changes.gpkg                             0.61 MB
t1_buildings.tif                         0.10 MB
t2_buildings.tif                         0.13 MB
```

在交互式地图上显示保存的变化图。

In [ ]:
change_map_path = os.path.join(out_folder, "change_map.tif")
geoai.view_raster(
    change_map_path,
    nodata=0,
    cmap="Reds",
    opacity=0.8,
    basemap=naip_2019_path,
    backend="ipyleaflet",
)

## 关键要点

1. 变化检测识别同一区域多时相图像之间的有意义差异。

2. 像图像差分和CVA这样的传统方法是无监督的且可解释的，但独立处理每个像素。

3. 深度学习孪生网络在特征空间中比较图像对，区分真实变化与辐射变化。

4. ChangeStar联合执行变化检测和语义分割，为两个时间步生成变化图和建筑轮廓。

5. 多个模型变体和可调阈值控制灵敏度和特异性之间的平衡。

6. The `geoai` package provides a complete change detection workflow with georeferenced GeoTIFF and vector polygon output.